# 01 — Data Quality (Phase 1)

การตรวจสอบคุณภาพข้อมูล OHLCV ของสัญญา S50 ที่ดึงผ่าน tvkit ในเฟส 1

**ROADMAP §1.5 deliverables** — heatmap ของ missing candles, การกระจายของ return รายปี / รายเซสชัน, การเปลี่ยนแปลงของ volume / open interest ข้าม rollover, และการกระจายของ spread

All code cells are English; markdown cells follow the csm-set convention of Thai narrative.

In [ ]:
from __future__ import annotations

import polars as pl

from tfex_s50_multi_tf_swing.config.settings import get_settings
from tfex_s50_multi_tf_swing.data import ParquetStore, SessionCalendar, Validator

settings = get_settings()
store = ParquetStore(settings.data_dir)
validator = Validator(calendar=SessionCalendar(roll_offset_days=settings.roll_offset_days))

## 1. Missing-candle heatmap per session

In [ ]:
# Read 5m continuous and bucket gaps by (date, session).
df = store.read_continuous("5m")
df.head()

## 2. Return distribution by year / by session

In [ ]:
# Compute log returns and group by year / by session.
returns = df.with_columns(
    (pl.col("close").cast(pl.Float64).log() - pl.col("close").cast(pl.Float64).shift(1).log()).alias("ret")
).drop_nulls("ret")
returns.describe()

## 3. Volume / open-interest evolution across rollovers

In [ ]:
df.group_by("contract_at_time").agg(pl.col("volume").sum().alias("total_volume")).sort("contract_at_time")

## 4. Spread distribution

In [ ]:
df.with_columns(
    ((pl.col("high").cast(pl.Float64) - pl.col("low").cast(pl.Float64)) / pl.col("close").cast(pl.Float64)).alias("spread_frac")
).select("spread_frac").describe()